# Correct PRIDE file endpoint and verified rhoptry publication linkage


The initial alternate file endpoint returned HTTP success with an empty body; it is not evidence of an empty deposit. Use the project-files endpoint already used by the repository fetcher. NCBI UpdateOf establishes the journal/preprint relationship, not supplementary-table equivalence.

In [1]:
import urllib.request,json,hashlib,xml.etree.ElementTree as ET
from pathlib import Path
output=Path('results/host_candidate_deposits_2026_10_07')
url='https://www.ebi.ac.uk/pride/ws/archive/v2/projects/PXD005200/files?pageSize=1000&page=0'
try:
    request=urllib.request.Request(url, headers={'User-Agent':'starplast-dataset-review/1'})
    with urllib.request.urlopen(request, timeout=25) as response:
        data=response.read(8 << 20)
    records=json.loads(data)
    (output/'rbc_project_files.json').write_bytes(data)
    result={'url':url,'status':'retrieved_valid_json','bytes':len(data),'sha256':hashlib.sha256(data).hexdigest(),'records':len(records)}
except Exception as exc:
    result={'url':url,'status':'failed','error':str(exc)}
    records=[]
(output/'corrected_file_request.json').write_text(json.dumps(result,indent=2)+'\n')
[(r.get("fileName"),r.get("fileSizeBytes"),r.get("fileCategory"),r.get("publicFileLocations")) for r in records if r.get("fileCategory",{}).get("value") != "RAW"]

[('MaxQuant_Output_4samples_3enzymes.zip', 28401219, {'@type': 'CvParam', 'cvLabel': 'PRIDE', 'accession': 'PRIDE:0000408', 'name': 'Search engine output file URI', 'value': 'SEARCH'}, [{'@type': 'CvParam', 'cvLabel': 'PRIDE', 'accession': 'PRIDE:0000469', 'name': 'FTP Protocol', 'value': 'ftp://ftp.pride.ebi.ac.uk/pride/data/archive/2017/08/PXD005200/MaxQuant_Output_4samples_3enzymes.zip'}, {'@type': 'CvParam', 'cvLabel': 'PRIDE', 'accession': 'PRIDE:0000468', 'name': 'Aspera Protocol', 'value': 'prd_ascp@fasp.ebi.ac.uk:pride/data/archive/2017/08/PXD005200/MaxQuant_Output_4samples_3enzymes.zip'}])]

In [2]:
root=ET.parse(output/'rhoptry_pubmed.xml').getroot()
links=[{'type':c.attrib.get('RefType'),'pmid':c.findtext('PMID'),'reference':c.findtext('RefSource')} for c in root.findall('.//CommentsCorrections')]
(output/'rhoptry_publication_linkage.json').write_text(json.dumps({'links':links,'supplement_equivalence':'unverified'},indent=2)+'\n')
links

[{'type': 'UpdateOf', 'pmid': '41279910', 'reference': 'bioRxiv. 2025 Oct 21:2025.10.16.682961. doi: 10.1101/2025.10.16.682961.'}]